In [ ]:
import sys
import os
import pandas as pd

src_path = os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)

from dense_index import DenseIndex
from sentence_transformers import SentenceTransformer
import citation_utils
import metric_utils
import reranker_utils
import rrf

In [ ]:
court_consideration_df = pd.read_csv("../data/court_considerations.csv")
court_consideration_d = dict(zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist()))

law_df = pd.read_csv("../data/laws_de.csv")
law_d = dict(zip(law_df['citation'].tolist(), law_df['text'].tolist()))

court_doc = [{'citation':citation, 'text':text} for citation,text in zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist())]
law_doc = [{'citation':citation, 'text':text} for citation,text in zip(law_df['citation'].tolist(), law_df['text'].tolist())]

_d = {}
for _, row in test_df.iterrows():
    if row['query_id'] not in _d:
        _d[row['query_id']] = [row['query']]
    else:
        _d[row['query_id']].append(row['query'])
test_dict = {k: v for k, v in sorted(_d.items())}


print("data loaded.")

In [ ]:
from FlagEmbedding import FlagReranker

model = SentenceTransformer("/root/.cache/modelscope/hub/models/Qwen/Qwen3-Embedding-0___6B", model_kwargs={"torch_dtype": "float16"})

dense_index = DenseIndex(model, "../data/processed/_dense_court", court_doc)

reranker = FlagReranker('../ft_data/merged_reranker', use_fp16=True, normalize=True) # Setting use_fp16 to True speeds up computation with a slight performance degradation


In [ ]:
RECALL_COUNT=1000
RERANK_COUNT=200
NN = 10

id_l = []
citation_l = []
for query_id, query_l in tqdm(test_dict.items(), total=len(test_dict)):
    ranked_l_l = []
    for query in query_l:
        court_dense_search_l = dense_index.search_with_score(query, RECALL_COUNT)
        ranked_l_l.append([c['citation'] for c,_ in court_dense_search_l])

    rrf_result = rrf.compute2_with_score(ranked_l_l, k=60, top_k=1000)

    rrf_doc_result = []
    for citation, score in rrf_result:
        if citation in court_consideration_d:
            rrf_doc_result.append({'citation':citation, 'text':court_consideration_d[citation]})
        elif citation in law_d:
            rrf_doc_result.append({'citation':citation, 'text':law_d[citation]})

    first_layer = rerank_utils.rerank_by_dense_batch_chunked(reranker, query, rrf_doc_result, RERANK_COUNT, 20, 384, 128)

    second_layer = citation_utils.compute_citation_score_with_sentence_pos(first_layer, decay="reciprocal")

    hits = []
    for citation, score in second_layer:
        if citation in court_consideration_d:
            hits.append({'citation':citation, 'text':court_consideration_d[citation]})
        elif citation in law_d:
            hits.append({'citation':citation, 'text':law_d[citation]})

    print("first_layer.len:", len(first_layer),"second_layer.len:", len(second_layer), "hits.len:", len(all_hits))

    citations = [r['citation'] for r in hits] # hits is sorted 
    id_l.append(query_id)
    citation_l.append(';'.join(citations))
    print(query_id, len(hits))

In [ ]:
query_id_l = []
predicted_citations_l = []
for query_id, predicted_citations in zip(id_l, citation_l):
    query_id_l.append(query_id)
    predicted_citations_l.append(';'.join(predicted_citations.split(';')[:30]))
sub_df = pd.DataFrame({'query_id':query_id_l, 'predicted_citations':predicted_citations_l})
sub_df.to_csv("../output/submission.csv", index=False)